# California Housing Prices — Exploratory Data Analysis

This notebook performs an exploratory data analysis (EDA) of the California Housing Prices dataset.

The goal of this analysis is to understand the structure, quality, distribution, and relationships within the dataset before building a machine learning model.

## Dataset

The dataset was obtained from Kaggle:

[California Housing Prices — Kaggle](https://www.kaggle.com/datasets/camnugent/california-housing-prices)

This dataset contains information about housing districts in California based on data from the 1990 U.S. Census.

## Objectives

- Understand the structure of the dataset
- Inspect data types and basic statistics
- Identify missing values and duplicate records
- Analyze feature distributions (numerical and categorical)
- Detect potential outliers
- Explore correlations between features and the target variable
- Investigate relationships between important features and house prices, including geographic patterns
- Identify data quality issues (e.g. capped values) that affect modeling
- Identify useful insights for the modeling stage

## 1. Import Libraries

We import the libraries required for data manipulation, numerical analysis, and visualization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_style("whitegrid")

## 2. Load Dataset

We load the dataset into a pandas DataFrame for further analysis.

In [ ]:
df = pd.read_csv("../data/housing.csv")

## 3. Initial Data Inspection

Before performing detailed analysis, we inspect the basic structure and contents of the dataset.

In [ ]:
print("Shape:", df.shape)

df.head()

In [ ]:
df.info()

In [ ]:
df.columns

## 4. Missing Values

Missing values can affect data analysis and machine learning models.

We identify the number and percentage of missing values in each feature.

In [ ]:
df.isnull().sum()

In [ ]:
missing_percentage = df.isnull().mean() * 100

missing_percentage.sort_values(ascending=False)

## 5. Duplicate Records

We check whether the dataset contains duplicate rows.

In [ ]:
df.duplicated().sum()

In [ ]:
df[df.duplicated()]

## 6. Descriptive Statistics

Descriptive statistics provide an overview of the numerical features, including their count, mean, standard deviation, minimum, maximum, and quartiles.

`ocean_proximity` is the only categorical column in the dataset, so we describe it separately.

In [ ]:
df.describe()

In [ ]:
# Categorical feature summary
df.describe(include="object")

In [ ]:
df["ocean_proximity"].value_counts()

## 7. Numerical Feature Distributions

We visualize the distributions of numerical features to understand their range, spread, skewness, and potential unusual values.

In [ ]:
numerical_features = df.select_dtypes(include=np.number).columns

df[numerical_features].hist(
    figsize=(15, 12),
    bins=30
)

plt.tight_layout()
plt.show()

## 8. Categorical Feature Distribution

We look at the distribution of `ocean_proximity`, since it was excluded from the numerical analysis above but can still be informative for the target variable.

In [ ]:
plt.figure(figsize=(8, 5))

sns.countplot(
    data=df,
    x="ocean_proximity",
    order=df["ocean_proximity"].value_counts().index
)

plt.title("Count of Districts by Ocean Proximity")
plt.xlabel("Ocean Proximity")
plt.ylabel("Count")

plt.show()

## 9. Target Variable Distribution

The target variable for this project is `median_house_value`.

We examine its distribution to understand the range and overall shape of house prices.

**Note:** this dataset is known to have `median_house_value` capped at $500,001 — districts with a "true" value above that threshold were recorded at the cap. We check for this below, since it affects how the target should be handled during modeling.

In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(
    data=df,
    x="median_house_value",
    kde=True
)

plt.title("Distribution of Median House Value")
plt.xlabel("Median House Value")
plt.ylabel("Frequency")

plt.show()

In [ ]:
capped_value = df["median_house_value"].max()
capped_count = (df["median_house_value"] == capped_value).sum()
capped_pct = capped_count / len(df) * 100

print(f"Max median_house_value: {capped_value}")
print(f"Districts at the cap: {capped_count} ({capped_pct:.2f}% of the dataset)")

If a non-trivial share of districts sit exactly at the maximum value, this is a **capping artifact**, not a natural outlier. Options to consider for modeling: drop capped rows, treat the task as censored regression, or clearly note the ceiling effect when evaluating predictions near the cap.

The same capping pattern is worth checking for `housing_median_age` (commonly capped at 52).

In [ ]:
capped_age = df["housing_median_age"].max()
capped_age_count = (df["housing_median_age"] == capped_age).sum()

print(f"Max housing_median_age: {capped_age}")
print(f"Districts at the cap: {capped_age_count} ({capped_age_count / len(df) * 100:.2f}% of the dataset)")

## 10. Outlier Detection

Boxplots are used to identify potential outliers in numerical features.

An outlier is not necessarily an incorrect observation. Potential outliers should be investigated before deciding whether they should be removed or transformed.

**Note:** the numerical features here have very different scales (e.g. `population` can be in the tens of thousands while `housing_median_age` tops out around 50), so plotting them on one shared axis makes most boxes unreadable. We plot each feature in its own subplot instead.

In [ ]:
n_cols = 3
n_rows = int(np.ceil(len(numerical_features) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(numerical_features):
    sns.boxplot(data=df, y=col, ax=axes[i])
    axes[i].set_title(col)

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

## 11. Geographic Distribution

California housing prices are strongly tied to location. Plotting `latitude` and `longitude`, colored by `median_house_value`, gives geographic context to every other finding in this notebook (coastal areas, the Bay Area, and greater LA are expected to stand out).

In [ ]:
plt.figure(figsize=(10, 8))

scatter = plt.scatter(
    df["longitude"],
    df["latitude"],
    c=df["median_house_value"],
    cmap="viridis",
    s=df["population"] / 100,
    alpha=0.4
)

plt.colorbar(scatter, label="Median House Value")
plt.title("Geographic Distribution of Median House Value")
plt.xlabel("Longitude")
plt.ylabel("Latitude")

plt.show()

## 12. Correlation Analysis

We examine the linear correlation between numerical features.

Correlation can help identify relationships between features and the target variable.

Correlation does not imply causation.

In [ ]:
correlation_matrix = df[numerical_features].corr()

correlation_matrix

In [ ]:
plt.figure(figsize=(12, 8))

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm"
)

plt.title("Correlation Matrix")

plt.tight_layout()
plt.show()

### Correlation with Target

We examine the correlation of each numerical feature with `median_house_value` to identify features that have stronger linear relationships with the target.

In [ ]:
target_correlation = (
    correlation_matrix["median_house_value"]
    .sort_values(ascending=False)
)

target_correlation

## 13. Feature vs Target Analysis

We visualize the relationship between selected features and the target variable.

This helps us understand whether the relationships appear linear, nonlinear, weak, or strong.

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x="median_income",
    y="median_house_value",
    alpha=0.3
)

plt.title("Median Income vs Median House Value")
plt.xlabel("Median Income")
plt.ylabel("Median House Value")

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x="housing_median_age",
    y="median_house_value",
    alpha=0.3
)

plt.title("Housing Median Age vs Median House Value")
plt.xlabel("Housing Median Age")
plt.ylabel("Median House Value")

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x="population",
    y="median_house_value",
    alpha=0.3
)

plt.title("Population vs Median House Value")
plt.xlabel("Population")
plt.ylabel("Median House Value")

plt.show()

We also look at `median_house_value` broken down by `ocean_proximity`, since it's the one feature not captured by the correlation matrix above.

In [ ]:
plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df,
    x="ocean_proximity",
    y="median_house_value",
    order=df.groupby("ocean_proximity")["median_house_value"].median().sort_values(ascending=False).index
)

plt.title("Median House Value by Ocean Proximity")
plt.xlabel("Ocean Proximity")
plt.ylabel("Median House Value")

plt.show()

## 14. Feature Engineering Exploration

Raw counts like `total_rooms` and `total_bedrooms` are less meaningful on their own than they are as ratios per household. We construct a few candidate features and check whether they correlate more strongly with the target than the originals.

In [ ]:
df["rooms_per_household"] = df["total_rooms"] / df["households"]
df["bedrooms_per_room"] = df["total_bedrooms"] / df["total_rooms"]
df["population_per_household"] = df["population"] / df["households"]

engineered_corr = df[[
    "median_house_value",
    "rooms_per_household",
    "bedrooms_per_room",
    "population_per_household"
]].corr()["median_house_value"].sort_values(ascending=False)

engineered_corr

These engineered ratios are candidates to carry forward into preprocessing — compare their correlation with the target against the raw `total_rooms`/`total_bedrooms`/`population` figures from Section 12.

## 15. EDA Findings

### Dataset Structure
- The dataset contains multiple numerical features describing housing districts in California, plus one categorical feature (`ocean_proximity`).
- `median_house_value` is used as the target variable.

### Data Quality
- Missing values were identified (concentrated in `total_bedrooms`) and should be handled during preprocessing.
- Duplicate records were checked.
- `median_house_value` is capped at $500,001 — a meaningful share of districts sit exactly at this value, which is a data quality/censoring issue rather than a natural outlier. `housing_median_age` shows a similar capping pattern at its maximum.

### Feature Distributions
- Several numerical features have skewed distributions.
- `ocean_proximity` is unevenly distributed, with a small number of districts in the `ISLAND` category.
- Some features contain potential outliers that should be investigated before modeling.

### Correlation
- `median_income` shows a relatively strong positive relationship with `median_house_value`.
- Other raw numerical features show weaker linear relationships with the target; engineered ratios (`rooms_per_household`, `bedrooms_per_room`, `population_per_household`) correlate more strongly than the raw counts they're derived from.
- Other features show weaker linear relationships with the target.

### Feature vs Target
- The relationship between `median_income` and `median_house_value` appears to be particularly important.
- `ocean_proximity` shows a clear association with price (coastal categories trend higher than `INLAND`).
- Geographic position (`latitude`/`longitude`) reveals clear regional price clusters (e.g. coastal and Bay Area districts), which the correlation matrix alone does not capture.
- Some feature-target relationships do not appear to be strictly linear.

### Modeling Considerations
- Missing values need to be handled before model training.
- The capped target values need an explicit decision (drop, censor-aware modeling, or documented caveat).
- Numerical features may require preprocessing (e.g. scaling, log-transforms for skewed features) depending on the selected model.
- `ocean_proximity` needs categorical encoding (e.g. one-hot).
- Potential outliers should be investigated rather than automatically removed.
- Engineered ratio features are worth carrying into the modeling stage.
- Feature relationships, including geographic structure, should be considered when selecting and evaluating regression models.

## 16. Modeling Considerations

Based on the EDA, the following considerations should be addressed before model training:

1. Handle missing values appropriately (e.g. impute `total_bedrooms`).
2. Decide how to handle the capped `median_house_value` records.
3. Review potential outliers.
4. Encode the `ocean_proximity` categorical feature.
5. Add engineered features (`rooms_per_household`, `bedrooms_per_room`, `population_per_household`).
6. Select relevant features.
7. Apply appropriate preprocessing to numerical features (e.g. scaling, log-transform for skewed distributions).
8. Split the dataset into training and testing sets.
9. Evaluate the regression model using appropriate metrics such as MAE, MSE, and RMSE.